In [ ]:
import numpy as np 
import pandas as pd
import re

In [ ]:
# get a table summarising how prophylactic agents were grouped

def get_prophylaxis_combinations(df, prophylaxis_raw_col='prophylaxis', prophylaxis_col='prophylaxis_group'):
    
    # get the unique combinations of prophylactic agents
    combinations = df[prophylaxis_col].unique()
    
    # create a summary table
    summary_table = pd.DataFrame(columns=['Combination', 'Count'])
    
    for combo in combinations:
        count = (df[prophylaxis_col] == combo).sum()
        summary_table = summary_table.append({'Combination': combo, 'Count': count}, ignore_index=True)
    
    return summary_table


In [ ]:
# How many patients from the df had prior positive EKP microbiology cultures?

def count_prior_positive_cultures(df, culture_col='prior_positive_culture'):

    # Count the number of patients with prior positive cultures
    count_positive = (df[culture_col] == True).sum()
    
    return count_positive

In [1]:
# helper functions 

def classify_mic_result(x, s_breakpoint, r_breakpoint):
    if pd.isna(x):
        return np.nan

    # Keep already-standardised categorical results
    x_str = str(x).strip().lower()

    if "resistant" in x:
        return "R"

    if "intermediate" in x:
        return "I"

    if "susceptible" in x or "sensitive" in x:
        return "S"

    if "optimised dosing" in x:
        return "I"

    op, mic_value = parse_mic(x)

    if mic_value is None:
        return np.nan

    # Exact or upper-bound MICs
    if op in ["=", "<=", "<"]:
        if mic_value <= s_breakpoint:
            return "S"
        elif mic_value > r_breakpoint:
            return "R"
        else:
            return "I"

    # Lower-bound MICs
    if op in [">", ">="]:
        if mic_value > r_breakpoint:
            return "R"
        elif mic_value <= s_breakpoint:
            # This is ambiguous: e.g. >0.25 when S <=0.25
            return np.nan
        else:
            return "I"

    return np.nan



def parse_mic(x):
    if pd.isna(x):
        return None, None

    x = str(x).strip().lower()

    match = re.match(r"^(<=|>=|<|>)?\s*([0-9]*\.?[0-9]+)$", x)  # captures optional operator
    if not match:
        return None, None

    operator = match.group(1) or "="
    value = float(match.group(2))

    return operator, value

In [ ]:
def label_esbl_status_eucast(row, ceph_r_set, ceph_s_set):
    # With the RAST method, ESBL production in E. coli, K. pneumoniae can be detected
    # by using cefotaxime and ceftazidime screening cutoff values at 4, 6, 8 and 16-20 hours.
    # Test both cefotaxime and ceftazidime with and without clavulanic acid.

    clav_breakpoints = (8, 8)

    # Extract sensitivity results for each antibiotic
    cefo = row.get('cefotaxime')
    cefta = row.get('ceftazidime')
    marker = row.get('esbl markers (ss = present)')

    clavulanic_acid = row.get('co-amoxiclav')
    augmentin = row.get('augmentin')  # fallback if co-amoxiclav is missing

    # replace the unknown clavulanic acid result with augmentin if available
    if pd.isna(clavulanic_acid) and not pd.isna(augmentin):
        clavulanic_acid = augmentin
        
    clavulanic_acid_parsed = parse_mic(clavulanic_acid) if not pd.isna(clavulanic_acid) else (np.nan, np.nan)
    clavulanic_acid = classify_mic_result(clavulanic_acid, clav_breakpoints[0], clav_breakpoints[1]) if not pd.isna(clavulanic_acid) else np.nan

    # Confirmed ESBL logic (3GCR resistance pattern or ESBL marker = sensitive)
    confirmed_esbl = (
        (cefo in ceph_r_set or cefta in ceph_r_set) or
        (marker in ['sensitive', 'susceptible',
                    'susceptible with optimised dosing, refer to antimicrobial policy']) and
                    
                    if clavulanic_acid:
                        clavulanic_acid in ['S', 'I']: 
    )

    # Non-ESBL logic (3GCS sensitivity or ESBL marker = resistant)
    non_esbl = (
        (cefo in ceph_s_set and cefta in ceph_s_set) or
        (marker == 'resistant')
    )

    if confirmed_esbl:
        return 'ESBL'
    elif non_esbl:
        return 'non-ESBL'
    else:
        return 'unknown'  # fallback for ambiguous or missing data